# PBL5 - 2-Stage Damage Detection (chieu 27/04)

## Pipeline 2-stage tren dataset DA ENHANCED
- **Stage 1**: YOLOv8m binary detector -> tim VUNG hu hai (1 class = "damage")
- **Stage 2**: YOLOv8m classifier -> phan loai 8 LOAI hu hai tren crop
- **Combined**: stage1 detect bbox -> crop -> stage2 classify -> output {bbox + class + conf}

## Diem moi so voi ban truoc
- Dataset DA ENHANCED:
  - **CLAHE** (local contrast tren L channel LAB) -> texture (fold/peel/dirt) ro hon
  - **Unsharp mask** -> edge (scratch/fold) sac net
  - **HSV boost selective** -> mau bat thuong (burn/staning) noi bat
- Dataset bundled them **labels_8class/** -> tinh duoc combined mAP@50 chinh xac
- Train tu dau (yolov8m.pt + yolov8m-cls.pt) tren data enhanced

## Setup Kaggle
1. **Add Data** -> upload zip `data_26_04_2stage_2345_9011.zip` (~2.2 GB) hoac folder da extract
2. **Settings**: Accelerator = **GPU T4 x2**, Internet ON
3. **Run All**

## Thoi gian uoc tinh (T4 x2)
- Stage 1 (yolov8m, 200 ep, imgsz 832): ~2.5 - 3h
- Stage 2 (yolov8m-cls, 100 ep, imgsz 224): ~1h
- Combined eval + visualize: ~10 phut

## Sau khi train xong
- **Save Version** (Kaggle menu) de luu vinh vien
- Hoac tai `results_2stage_*.zip` tu Output panel
- Models: `runs/stage1/<name>/weights/best.pt` + `runs/stage2/<name>/weights/best.pt`

## 0. Setup environment

In [ ]:
import os, sys, zipfile, shutil, json
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

# Tat noise tu TF/absl/cuDNN tren Kaggle
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['GRPC_VERBOSITY'] = 'ERROR'
os.environ['ABSL_LOGGING_VERBOSITY'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import warnings; warnings.filterwarnings('ignore')

# Uninstall Ray: tranh bug callback raytune cua Ultralytics 8.3.39 vs Ray API moi
!pip uninstall -y ray 2>/dev/null | tail -1
!pip install -q -U ultralytics==8.3.39

import torch
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    free, tot = torch.cuda.mem_get_info(i)
    print(f'  GPU{i}: {torch.cuda.get_device_name(i)}  ({tot/1e9:.1f} GB)')

## 1. Tim/extract dataset 2-stage

In [ ]:
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')

def find_2stage_root(base: Path):
    """Tim folder co cau truc stage1/images/train + stage2/train/<class>/."""
    for cand in base.rglob('stage1'):
        if (cand / 'images' / 'train').is_dir() and (cand.parent / 'stage2' / 'train').is_dir():
            return cand.parent
    return None

DATA_ROOT = find_2stage_root(INPUT)

if DATA_ROOT is not None:
    print(f'Found extracted dataset: {DATA_ROOT}')
else:
    zips = list(INPUT.rglob('data_26_04_2stage_*.zip')) or list(INPUT.rglob('*2stage*.zip'))
    assert zips, 'Khong tim thay dataset 2-stage. Hay Add Data zip hoac folder.'
    ZIP = zips[0]
    print(f'Extracting {ZIP} ({ZIP.stat().st_size/1e6:.1f} MB)...')
    extract_root = WORK / 'data_extracted'
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(extract_root)
    DATA_ROOT = find_2stage_root(extract_root)
    assert DATA_ROOT is not None, f'Sau extract khong tim thay stage1/stage2 trong {extract_root}'
    print(f'Extracted to: {DATA_ROOT}')

STAGE1_DIR = DATA_ROOT / 'stage1'
STAGE2_DIR = DATA_ROOT / 'stage2'

# Verify
for s in ('train', 'val', 'test'):
    n_s1 = len(list((STAGE1_DIR / 'images' / s).iterdir()))
    n_s2 = sum(len(list(d.iterdir())) for d in (STAGE2_DIR / s).iterdir() if d.is_dir())
    has_8c = (STAGE1_DIR / 'labels_8class' / s).is_dir()
    print(f'  {s:5s}: stage1={n_s1:5d} images | stage2={n_s2:5d} crops | 8class_GT={has_8c}')

In [ ]:
# Stage 1 yaml: input la read-only, ghi yaml o /kaggle/working
import yaml
STAGE1_YAML = WORK / 'stage1.yaml'
yaml.safe_dump({
    'path': STAGE1_DIR.as_posix(),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 1,
    'names': ['damage'],
}, open(STAGE1_YAML, 'w'), sort_keys=False)
print(f'Wrote: {STAGE1_YAML}')
print(STAGE1_YAML.read_text())

## 2. Class distribution Stage 2

In [ ]:
CLASS_NAMES = ['material_loss', 'peel', 'scratch', 'fold',
               'writing_marks', 'dirt', 'staning', 'burn_marks']

print(f'  {"class":<17} {"train":>8} {"val":>8} {"test":>8}')
print(f'  {"-"*17} {"-"*8} {"-"*8} {"-"*8}')
for cname in CLASS_NAMES:
    counts = []
    for split in ('train', 'val', 'test'):
        d = STAGE2_DIR / split / cname
        counts.append(len(list(d.iterdir())) if d.is_dir() else 0)
    print(f'  {cname:<17} {counts[0]:>8} {counts[1]:>8} {counts[2]:>8}')
totals = [sum(len(list((STAGE2_DIR / split / c).iterdir())) for c in CLASS_NAMES if (STAGE2_DIR / split / c).is_dir())
          for split in ('train', 'val', 'test')]
print(f'  {"TOTAL":<17} {totals[0]:>8} {totals[1]:>8} {totals[2]:>8}')

## 3. STAGE 1 - YOLOv8 binary detector

**Muc tieu**: Tim VUNG hu hai (khong can phan biet loai)

**Bi quyet:**
- Augment manh hon 1-stage (binary -> khong sai class) -> generalize cao
- imgsz 832 de bat scratch/dirt nho
- yolov8m: tradeoff toc do/accuracy tot

In [ ]:
# ====== STAGE 1 CONFIG ======
S1_MODEL    = 'yolov8m.pt'
S1_EPOCHS   = 200
S1_BATCH    = 12
S1_IMGSZ    = 832
S1_PATIENCE = 40
S1_LR0      = 0.001
S1_DEVICE   = 0

# Auto: dung dual GPU neu co
if torch.cuda.device_count() >= 2:
    S1_DEVICE = '0,1'
    S1_BATCH = S1_BATCH * 2
    print(f'Dual GPU detected -> device={S1_DEVICE}, batch={S1_BATCH}')

S1_RUN = f"stage1_{datetime.now().strftime('%m%d_%H%M')}_{S1_MODEL.replace('.pt','')}"
S1_PROJECT = '/kaggle/working/runs/stage1'
print('Stage 1 run name:', S1_RUN)

In [ ]:
from ultralytics import YOLO

model_s1 = YOLO(S1_MODEL)

# Disable broken raytune callback (Ultralytics 8.3.39 + Ray API moi tren Kaggle)
for hook in list(model_s1.callbacks.keys()):
    model_s1.callbacks[hook] = [
        cb for cb in model_s1.callbacks[hook]
        if 'raytune' not in getattr(cb, '__module__', '')
    ]

model_s1.train(
    data=str(STAGE1_YAML),
    epochs=S1_EPOCHS,
    batch=S1_BATCH,
    imgsz=S1_IMGSZ,
    device=S1_DEVICE,
    name=S1_RUN,
    project=S1_PROJECT,
    exist_ok=True,
    patience=S1_PATIENCE,
    workers=4,
    seed=42,
    deterministic=True,

    # Optimizer
    optimizer='AdamW',
    lr0=S1_LR0,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    momentum=0.937,
    cos_lr=True,

    # AUGMENTATION manh (binary -> khong sai class)
    hsv_h=0.015,
    hsv_s=0.50,
    hsv_v=0.40,
    degrees=10.0,
    translate=0.10,
    scale=0.40,
    shear=3.0,
    perspective=0.0,
    flipud=0.20,
    fliplr=0.50,
    mosaic=0.80,
    mixup=0.15,
    copy_paste=0.30,
    erasing=0.30,

    # Loss
    cls=0.5,
    box=7.5,
    dfl=1.5,

    val=True, plots=True, save=True,
    save_period=25, close_mosaic=15, amp=True,
    cache=False, rect=False, nbs=64,
)

S1_BEST = Path(S1_PROJECT) / S1_RUN / 'weights' / 'best.pt'
S1_LAST = Path(S1_PROJECT) / S1_RUN / 'weights' / 'last.pt'
print(f'\n[Stage 1 done]')
print(f'  Best: {S1_BEST}')
print(f'  Last: {S1_LAST}')

### 3.1 Validate Stage 1 tren val + test (voi TTA)

In [ ]:
model_s1 = YOLO(str(S1_BEST))

s1_metrics = {}
for split in ('val', 'test'):
    print(f'\n=== Stage 1 - {split.upper()} (TTA) ===')
    m = model_s1.val(
        data=str(STAGE1_YAML), split=split,
        batch=S1_BATCH, imgsz=S1_IMGSZ, device=S1_DEVICE,
        plots=True, save_json=True, conf=0.001, iou=0.6,
        augment=True,
        name=f's1_{split}_{S1_RUN}', project=S1_PROJECT, exist_ok=True,
    )
    s1_metrics[split] = {
        'mAP50': float(m.box.map50),
        'mAP50_95': float(m.box.map),
        'P': float(m.box.mp),
        'R': float(m.box.mr),
    }
    print(f'  mAP50:    {m.box.map50:.4f}')
    print(f'  mAP50-95: {m.box.map:.4f}')
    print(f'  P:        {m.box.mp:.4f}')
    print(f'  R:        {m.box.mr:.4f}')

print('\nSummary:')
for split, m in s1_metrics.items():
    print(f'  {split}: mAP50={m["mAP50"]:.4f}  mAP50-95={m["mAP50_95"]:.4f}  P={m["P"]:.4f}  R={m["R"]:.4f}')

## 4. STAGE 2 - YOLOv8 classifier (8-class)

**Muc tieu**: Phan loai loai hu hai tren crop 256x256 (training 224x224)

**Bi quyet:**
- Crop nho -> ResNet/EfficientNet-style features hieu qua
- yolov8m-cls (15M params, ImageNet pretrained)
- Augment vua phai (crop nho -> de bi distort qua)

In [ ]:
# ====== STAGE 2 CONFIG ======
S2_MODEL    = 'yolov8m-cls.pt'
S2_EPOCHS   = 100
S2_BATCH    = 64
S2_IMGSZ    = 224
S2_PATIENCE = 30
S2_LR0      = 0.001
S2_DEVICE   = 0

if torch.cuda.device_count() >= 2:
    S2_DEVICE = '0,1'
    S2_BATCH = S2_BATCH * 2
    print(f'Dual GPU detected -> device={S2_DEVICE}, batch={S2_BATCH}')

S2_RUN = f"stage2_{datetime.now().strftime('%m%d_%H%M')}_{S2_MODEL.replace('.pt','')}"
S2_PROJECT = '/kaggle/working/runs/stage2'
print('Stage 2 run name:', S2_RUN)

In [ ]:
model_s2 = YOLO(S2_MODEL)

for hook in list(model_s2.callbacks.keys()):
    model_s2.callbacks[hook] = [
        cb for cb in model_s2.callbacks[hook]
        if 'raytune' not in getattr(cb, '__module__', '')
    ]

# Stage 2 train tu folder structure: stage2/{train,val,test}/<class>/
model_s2.train(
    data=str(STAGE2_DIR),
    epochs=S2_EPOCHS,
    batch=S2_BATCH,
    imgsz=S2_IMGSZ,
    device=S2_DEVICE,
    name=S2_RUN,
    project=S2_PROJECT,
    exist_ok=True,
    patience=S2_PATIENCE,
    workers=4,
    seed=42,
    deterministic=True,

    # Optimizer
    optimizer='AdamW',
    lr0=S2_LR0,
    lrf=0.01,
    weight_decay=0.0001,
    warmup_epochs=3,
    momentum=0.937,
    cos_lr=True,

    # AUGMENTATION cho crop classification
    hsv_h=0.015,
    hsv_s=0.40,
    hsv_v=0.30,
    degrees=15.0,
    translate=0.05,
    scale=0.20,
    fliplr=0.50,
    flipud=0.10,
    erasing=0.25,
    auto_augment='randaugment',

    # Misc
    val=True, plots=True, save=True,
    save_period=20, amp=True,
)

S2_BEST = Path(S2_PROJECT) / S2_RUN / 'weights' / 'best.pt'
S2_LAST = Path(S2_PROJECT) / S2_RUN / 'weights' / 'last.pt'
print(f'\n[Stage 2 done]')
print(f'  Best: {S2_BEST}')
print(f'  Last: {S2_LAST}')

### 4.1 Validate Stage 2 standalone (top-1 / top-5 acc)

In [ ]:
model_s2 = YOLO(str(S2_BEST))

s2_metrics = {}
for split in ('val', 'test'):
    print(f'\n=== Stage 2 standalone - {split.upper()} ===')
    m = model_s2.val(
        data=str(STAGE2_DIR), split=split,
        batch=S2_BATCH, imgsz=S2_IMGSZ, device=S2_DEVICE,
        plots=True,
        name=f's2_{split}_{S2_RUN}', project=S2_PROJECT, exist_ok=True,
    )
    s2_metrics[split] = {'top1': float(m.top1), 'top5': float(m.top5)}
    print(f'  top-1 acc: {m.top1:.4f}')
    print(f'  top-5 acc: {m.top5:.4f}')

print('\nSummary:')
for split, m in s2_metrics.items():
    print(f'  {split}: top1={m["top1"]:.4f}  top5={m["top5"]:.4f}')

## 5. COMBINED PIPELINE - Stage 1 + Stage 2

**Pipeline**: anh -> Stage 1 detect bbox 'damage' -> crop + 20% padding -> Stage 2 classify -> {bbox + class + conf_combined}

**Combined confidence** = `s1_conf * s2_conf`

In [ ]:
import numpy as np
from PIL import Image

# IMPORTANT: predict() KHONG support multi-GPU -> luon dung 1 GPU cho inference
PREDICT_DEVICE = 0

# Load both models
model_s1 = YOLO(str(S1_BEST))
model_s2 = YOLO(str(S2_BEST))

# Stage 2 names: alphabet sorted (theo folder structure)
S2_NAMES = model_s2.names if hasattr(model_s2, 'names') else None
if isinstance(S2_NAMES, dict):
    S2_NAMES = [S2_NAMES[i] for i in sorted(S2_NAMES)]
print(f'Stage 2 class names: {S2_NAMES}')

# Map name -> global index theo CLASS_NAMES toan cuc
NAME_TO_GLOBAL = {n: i for i, n in enumerate(CLASS_NAMES)}

# Test images
TEST_IMG_DIR = STAGE1_DIR / 'images' / 'test'
test_imgs = sorted(TEST_IMG_DIR.iterdir())
print(f'Test images: {len(test_imgs)}')

In [ ]:
def crop_with_pad(im, x1, y1, x2, y2, pad=0.20, size=224):
    """Crop bbox + padding -> resize ve square voi pad mau xam."""
    W, H = im.size
    bw, bh = x2 - x1, y2 - y1
    px, py = bw * pad, bh * pad
    cx1 = max(0, int(x1 - px))
    cy1 = max(0, int(y1 - py))
    cx2 = min(W, int(x2 + px))
    cy2 = min(H, int(y2 + py))
    crop = im.crop((cx1, cy1, cx2, cy2))
    crop.thumbnail((size, size), Image.LANCZOS)
    bg = Image.new('RGB', (size, size), (128, 128, 128))
    bg.paste(crop, ((size - crop.width) // 2, (size - crop.height) // 2))
    return bg


def run_pipeline(img_path, conf_s1=0.10, iou_s1=0.5):
    """Stage 1 detect -> Stage 2 classify -> 8-class output."""
    r1 = model_s1.predict(img_path, conf=conf_s1, iou=iou_s1, imgsz=S1_IMGSZ,
                          device=PREDICT_DEVICE, save=False, verbose=False)[0]
    if r1.boxes is None or len(r1.boxes) == 0:
        return [], [], []
    boxes = r1.boxes.xyxy.cpu().numpy()
    s1_confs = r1.boxes.conf.cpu().numpy()

    with Image.open(img_path) as im:
        if im.mode != 'RGB':
            im = im.convert('RGB')
        crops = [crop_with_pad(im, *b, size=S2_IMGSZ) for b in boxes]
    r2_list = model_s2.predict(crops, imgsz=S2_IMGSZ, device=PREDICT_DEVICE,
                               save=False, verbose=False)
    classes, s2_confs = [], []
    for r2 in r2_list:
        probs = r2.probs.data.cpu().numpy() if r2.probs is not None else None
        if probs is None:
            classes.append(-1); s2_confs.append(0.0)
            continue
        cls_idx = int(np.argmax(probs))
        cls_name = S2_NAMES[cls_idx]
        global_idx = NAME_TO_GLOBAL.get(cls_name, -1)
        classes.append(global_idx)
        s2_confs.append(float(probs[cls_idx]))

    final_conf = s1_confs * np.array(s2_confs)
    return boxes.tolist(), classes, final_conf.tolist()


# Quick test 1 anh
boxes, cls, confs = run_pipeline(test_imgs[0])
print(f'Sample {test_imgs[0].name}: {len(boxes)} detections')
for b, c, f in zip(boxes[:5], cls[:5], confs[:5]):
    nm = CLASS_NAMES[c] if c >= 0 else 'UNKNOWN'
    print(f'  bbox={[int(x) for x in b]}  class={nm}  conf={f:.3f}')

### 5.1 Tinh combined mAP@50 tren test set

In [ ]:
from tqdm import tqdm

# Tim 8-class GT labels (uu tien stage1/labels_8class/test/ bundled trong zip)
def find_8class_gt(stage1_dir, input_root, work_root):
    p = stage1_dir / 'labels_8class' / 'test'
    if p.is_dir() and any(p.iterdir()):
        return p, stage1_dir / 'images' / 'test'
    for base in (input_root, work_root):
        for yml in base.rglob('data.yaml'):
            try:
                cfg = yaml.safe_load(open(yml))
                if cfg.get('nc') == 8 and (yml.parent / 'labels' / 'test').is_dir():
                    return yml.parent / 'labels' / 'test', yml.parent / 'images' / 'test'
            except: pass
    for base in (input_root, work_root):
        for cand in base.rglob('datatest'):
            lbl = cand / 'train' / 'labels'
            img = cand / 'train' / 'images'
            if lbl.is_dir() and img.is_dir() and any(lbl.iterdir()):
                return lbl, img
    return None, None

GT_LBL_TEST, GT_IMG_TEST = find_8class_gt(STAGE1_DIR, INPUT, WORK)
print(f'GT 8-class labels test: {GT_LBL_TEST}')
print(f'GT 8-class images test: {GT_IMG_TEST}')

if GT_LBL_TEST is None:
    print('\nWARN: khong tim thay GT 8-class -> bo qua mAP combined eval.')

In [ ]:
def yolo_to_xyxy(cx, cy, w, h, W, H):
    return ((cx - w/2)*W, (cy - h/2)*H, (cx + w/2)*W, (cy + h/2)*H)

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    a1 = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    a2 = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (a1 + a2 - inter + 1e-9)

if GT_LBL_TEST is not None:
    preds_per_class = {c: [] for c in range(8)}
    n_gt_per_class = Counter()

    for img_path in tqdm(test_imgs, desc='Combined eval'):
        boxes, classes, confs = run_pipeline(img_path, conf_s1=0.10)

        gt_lbl = GT_LBL_TEST / f'{img_path.stem}.txt'
        if not gt_lbl.exists():
            continue
        with Image.open(img_path) as im:
            W, H = im.size
        gts = []
        for line in gt_lbl.read_text().strip().split('\n'):
            parts = line.strip().split()
            if len(parts) != 5: continue
            try:
                c = int(parts[0]); cx, cy, w, h = map(float, parts[1:])
            except ValueError:
                continue
            gts.append((c, *yolo_to_xyxy(cx, cy, w, h, W, H)))
            n_gt_per_class[c] += 1
        used_gt = [False] * len(gts)

        order = np.argsort(-np.array(confs)) if confs else []
        for idx in order:
            pb, pc, pf = boxes[idx], classes[idx], confs[idx]
            if pc < 0:
                continue
            best_iou = 0; best_g = -1
            for gi, gt in enumerate(gts):
                if used_gt[gi] or gt[0] != pc:
                    continue
                io = iou(pb, gt[1:])
                if io > best_iou:
                    best_iou = io; best_g = gi
            tp = best_iou >= 0.5 and best_g >= 0
            if tp: used_gt[best_g] = True
            preds_per_class[pc].append((pf, tp))

    print(f'\n  {"class":<17} {"#GT":>6} {"#Pred":>6} {"AP@50":>8}')
    print(f'  {"-"*17} {"-"*6} {"-"*6} {"-"*8}')
    aps = []
    for c in range(8):
        preds = sorted(preds_per_class[c], key=lambda x: -x[0])
        n_gt = n_gt_per_class[c]
        if n_gt == 0:
            print(f'  {CLASS_NAMES[c]:<17} {0:>6} {len(preds):>6} {"N/A":>8}')
            continue
        tp = np.array([1 if p[1] else 0 for p in preds])
        fp = 1 - tp
        cum_tp = np.cumsum(tp); cum_fp = np.cumsum(fp)
        recall = cum_tp / max(n_gt, 1)
        precision = cum_tp / np.maximum(cum_tp + cum_fp, 1)
        ap = 0.0
        for r in np.linspace(0, 1, 11):
            mask = recall >= r
            if mask.any():
                ap += precision[mask].max() / 11
        aps.append(ap)
        print(f'  {CLASS_NAMES[c]:<17} {n_gt:>6} {len(preds):>6} {ap:>8.4f}')
    combined_map50 = float(np.mean(aps))
    print(f'\n  COMBINED mAP@50 (test set): {combined_map50:.4f}')

    report = {
        'timestamp': datetime.now().isoformat(),
        's1_run': S1_RUN, 's2_run': S2_RUN,
        's1_metrics': s1_metrics, 's2_metrics': s2_metrics,
        'combined_mAP50_test': combined_map50,
        'per_class_AP50': {CLASS_NAMES[c]: float(a) for c, a in zip(range(8), aps)},
    }
    report_path = WORK / f'report_{S1_RUN}_{S2_RUN}.json'
    json.dump(report, open(report_path, 'w'), indent=2)
    print(f'\nReport: {report_path}')

## 6. So sanh: Stage 2 standalone vs Combined per-class

In [ ]:
# Stage 2 standalone tren test crops -> top-1 acc per class
test_per_class = defaultdict(lambda: [0, 0])

for cname in CLASS_NAMES:
    cdir = STAGE2_DIR / 'test' / cname
    if not cdir.is_dir():
        continue
    crops = list(cdir.iterdir())
    if not crops:
        continue
    rs = model_s2.predict(crops, imgsz=S2_IMGSZ, device=PREDICT_DEVICE,
                          save=False, verbose=False)
    for r in rs:
        pred_idx = int(r.probs.top1)
        pred_name = S2_NAMES[pred_idx]
        test_per_class[cname][1] += 1
        if pred_name == cname:
            test_per_class[cname][0] += 1

print(f'  {"class":<17} {"S2_only_acc":>12} {"correct":>8} {"total":>8}')
print(f'  {"-"*17} {"-"*12} {"-"*8} {"-"*8}')
for cname in CLASS_NAMES:
    c, t = test_per_class[cname]
    acc = c / t if t else 0
    print(f'  {cname:<17} {acc:>12.4f} {c:>8} {t:>8}')

## 7. Visualize predictions tren test set

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(13)
samples = random.sample(test_imgs, 6)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for ax, img_path in zip(axes.flat, samples):
    boxes, classes, confs = run_pipeline(img_path, conf_s1=0.20)
    with Image.open(img_path) as im:
        ax.imshow(im)
    cmap = plt.get_cmap('tab10')
    for b, c, f in zip(boxes, classes, confs):
        if c < 0 or f < 0.10:
            continue
        rect = patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                                  linewidth=2, edgecolor=cmap(c), facecolor='none')
        ax.add_patch(rect)
        ax.text(b[0], b[1]-5, f'{CLASS_NAMES[c]} {f:.2f}',
                color='white', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', fc=cmap(c), ec='none'))
    ax.set_title(img_path.name[:30], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Export ONNX

In [ ]:
print('Exporting Stage 1 to ONNX...')
YOLO(str(S1_BEST)).export(format='onnx', imgsz=S1_IMGSZ, simplify=True, opset=12)

print('Exporting Stage 2 to ONNX...')
YOLO(str(S2_BEST)).export(format='onnx', imgsz=S2_IMGSZ, simplify=True, opset=12)

for d in (S1_BEST.parent, S2_BEST.parent):
    for f in d.glob('*.onnx'):
        print(f'  {f}  ({f.stat().st_size/1e6:.1f} MB)')

## 9. Goi gon ket qua tai ve

In [ ]:
OUT_ZIP = WORK / f'results_2stage_{datetime.now().strftime("%m%d_%H%M")}.zip'
with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for d in (Path(S1_PROJECT), Path(S2_PROJECT)):
        if not d.exists():
            continue
        for root, _, files in os.walk(d):
            for f in files:
                full = Path(root) / f
                zf.write(full, full.relative_to(WORK))
    for f in WORK.glob('report_*.json'):
        zf.write(f, f.name)

print(f'Output zip: {OUT_ZIP} ({OUT_ZIP.stat().st_size/1e6:.1f} MB)')
print('\nTai file nay tu Kaggle Output panel sau khi notebook chay xong.')
print('\nHoac click "Save Version" tren Kaggle de luu vinh vien notebook + outputs.')

## 10. Tom tat ket qua

Sau khi chay xong, ket qua se duoc luu o:
- `runs/stage1/<S1_RUN>/weights/best.pt` (binary detector)
- `runs/stage2/<S2_RUN>/weights/best.pt` (8-class classifier)
- `report_<S1_RUN>_<S2_RUN>.json` (metrics combined)
- `results_2stage_<timestamp>.zip` (toan bo packaging)

**Kien thuc cot loi:**
- Stage 1 + Stage 2 train tach roi -> de debug, generalize tot
- Image enhancement (CLAHE + unsharp + HSV boost) lam damage features noi bat
- Combined mAP@50 = metric chinh xac nhat de bao cao PBL